In [ ]:
!pip install transformers datasets peft accelerate bitsandbytes torch

# Part 1 – Understanding Model Types

## 1.1 Base Models vs Instruct Models

Understanding the difference is crucial before finetuning!

### Base Models (Pre-trained / Foundation Models)
- **Training**: Next-token prediction on massive text corpora
- **Behavior**: Completes text, continues patterns
- **Examples**: LLaMA-2-7b, GPT-2, Mistral-7B-v0.1
- **Use case**: Further training, research, custom finetuning

### Instruct Models (Chat / Aligned Models)
- **Training**: Base model + instruction finetuning + (often) RLHF
- **Behavior**: Follows instructions, answers questions, helpful assistant
- **Examples**: LLaMA-2-7b-chat, GPT-3.5-turbo, Mistral-7B-Instruct
- **Use case**: Direct deployment, chatbots, task completion

In [20]:
# Demonstrate the difference
from transformers import AutoTokenizer

# Example prompts showing different behaviors
base_prompt = "The capital of France is"
instruct_prompt = "What is the capital of France?"

print("=" * 60)
print("BASE MODEL BEHAVIOR:")
print("=" * 60)
print(f"Prompt: '{base_prompt}'")
print(f"Expected: 'Paris. What is the capital of Germany? Berlin...' (continues pattern)")
print()

print("=" * 60)
print("INSTRUCT MODEL BEHAVIOR:")
print("=" * 60)
print(f"Prompt: '{instruct_prompt}'")
print(f"Expected: 'The capital of France is Paris.' (answers question)")
print()

print("💡 Key Insight:")
print("   Base models are completion engines")
print("   Instruct models are question-answering assistants")

BASE MODEL BEHAVIOR:
Prompt: 'The capital of France is'
Expected: 'Paris. What is the capital of Germany? Berlin...' (continues pattern)

INSTRUCT MODEL BEHAVIOR:
Prompt: 'What is the capital of France?'
Expected: 'The capital of France is Paris.' (answers question)

💡 Key Insight:
   Base models are completion engines
   Instruct models are question-answering assistants


## 1.2 Chat Templates: Structuring Conversations

Chat/Instruct models need **structured input format** to work properly!

### What is a Chat Template?
A chat template wraps your conversation in special tokens that tell the model:
- Who is speaking (user, assistant, system)
- When a turn starts/ends
- When to generate vs stop

### Why Different Templates?
Each model family was trained with a specific format. Using the **wrong template** can:
- Confuse the model
- Generate poor responses
- Ignore instructions
- Not stop generating properly

In [14]:
# Common chat template formats
def show_chat_templates():
    """Display different chat template formats"""
    
    conversation = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is 2+2?"},
        {"role": "assistant", "content": "2+2 equals 4."},
        {"role": "user", "content": "What about 3+3?"}
    ]
    
    print("=" * 70)
    print("CHAT TEMPLATE EXAMPLES")
    print("=" * 70)
    
    # LLaMA-2 / Mistral format
    print("\n1. LLaMA-2 / Mistral Format:")
    print("-" * 70)
    llama_template = "<s>[INST] <<SYS>>\n{system}\n<</SYS>>\n\n{user1} [/INST] {assistant1} </s><s>[INST] {user2} [/INST]"
    print(llama_template.format(
        system="You are a helpful assistant.",
        user1="What is 2+2?",
        assistant1="2+2 equals 4.",
        user2="What about 3+3?"
    ))
    
    # ChatML format (OpenAI, Mistral-new)
    print("\n2. ChatML Format (OpenAI, newer models):")
    print("-" * 70)
    chatml = """<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is 2+2?<|im_end|>
<|im_start|>assistant
2+2 equals 4.<|im_end|>
<|im_start|>user
What about 3+3?<|im_end|>
<|im_start|>assistant"""
    print(chatml)
    
    # Alpaca format (simpler)
    print("\n3. Alpaca Format:")
    print("-" * 70)
    alpaca = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is 2+2?

### Response:
2+2 equals 4.

### Instruction:
What about 3+3?

### Response:"""
    print(alpaca)
    
    # Vicuna format
    print("\n4. Vicuna Format:")
    print("-" * 70)
    vicuna = """A chat between a curious user and an artificial intelligence assistant.

USER: What is 2+2?
ASSISTANT: 2+2 equals 4.</s>
USER: What about 3+3?
ASSISTANT:"""
    print(vicuna)
    
    print("\n" + "=" * 70)

show_chat_templates()

CHAT TEMPLATE EXAMPLES

1. LLaMA-2 / Mistral Format:
----------------------------------------------------------------------
<s>[INST] <<SYS>>
You are a helpful assistant.
<</SYS>>

What is 2+2? [/INST] 2+2 equals 4. </s><s>[INST] What about 3+3? [/INST]

2. ChatML Format (OpenAI, newer models):
----------------------------------------------------------------------
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is 2+2?<|im_end|>
<|im_start|>assistant
2+2 equals 4.<|im_end|>
<|im_start|>user
What about 3+3?<|im_end|>
<|im_start|>assistant

3. Alpaca Format:
----------------------------------------------------------------------
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is 2+2?

### Response:
2+2 equals 4.

### Instruction:
What about 3+3?

### Response:

4. Vicuna Format:
----------------------------------------------------------------------
A chat between a curious user

### Using Chat Templates in Code

In [22]:
# Proper way to use chat templates with tokenizers
def demonstrate_chat_template_usage():
    """Show how to properly apply chat templates"""
    
    # Example with a model that has chat template
    model_name = "mistralai/Mistral-7B-Instruct-v0.1"
    
    # Conversation in standard format
    messages = [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": "Explain quantum computing in simple terms."},
    ]
    
    print("Standard conversation format:")
    print(messages)
    print()
    
    # The tokenizer automatically applies the correct template!
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    formatted = tokenizer.apply_chat_template(messages, tokenize=False)
    print("After applying chat template:")
    print(formatted)
    print()
    
    print("💡 Pro Tip: Always use tokenizer.apply_chat_template()")
    print("   - Automatically uses correct format for the model")
    print("   - Handles special tokens properly")
    print("   - Prevents template mismatch errors")

demonstrate_chat_template_usage()

Standard conversation format:
[{'role': 'system', 'content': 'You are a helpful AI assistant.'}, {'role': 'user', 'content': 'Explain quantum computing in simple terms.'}]

After applying chat template:
<s> [INST] You are a helpful AI assistant.

Explain quantum computing in simple terms. [/INST]

💡 Pro Tip: Always use tokenizer.apply_chat_template()
   - Automatically uses correct format for the model
   - Handles special tokens properly
   - Prevents template mismatch errors


### Critical Rule for Finetuning:

**When finetuning an instruct model, you MUST:**
1. Use the same chat template it was trained with
2. Apply the template to your training data
3. Keep the special tokens consistent

**When training a base model into an instruct model:**
1. Choose a chat template (or design your own)
2. Apply consistently across all training data
3. Document which template you used!

## 1.3 Reasoning Tokens: Teaching Models to "Think"

Recent trend: Give models space to reason before answering!

### What are Reasoning Tokens?

Special tokens or patterns that:
1. Prompt the model to show its thinking process
2. Encourage step-by-step reasoning
3. Improve accuracy on complex problems

It's just a `<think></think>` tokens and the same instruction template!

# Part 2: Introduction to LLM Finetuning

## Why Finetune?
- Adapt pre-trained models to specific tasks/domains
- Improve performance on downstream tasks
- Customize behavior for specific use cases

## The Challenge: Full Finetuning is Expensive
- Modern LLMs have billions of parameters
- Full finetuning requires:
  - High memory (store all gradients)
  - Long training time
  - Expensive compute (multiple GPUs)

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import numpy as np

# Let's visualize the scale of the problem
def count_parameters(model):
    """Count trainable parameters in a model"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Example: Load a small model to demonstrate
model_name = "gpt2"  # 124M parameters - manageable for demo
print(f"Loading {model_name}...")

# Uncomment to load (skip in demo to save time)
# model = AutoModelForCausalLM.from_pretrained(model_name)
# print(f"Total parameters: {count_parameters(model):,}")

# For larger models:
print("\nParameter counts for reference:")
print("GPT-2: 124M parameters")
print("GPT-2 Medium: 355M parameters")
print("GPT-2 Large: 774M parameters")
print("LLaMA-7B: 7B parameters")
print("LLaMA-13B: 13B parameters")

/home/robinhad/Projects/ucu-nlp-course/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading gpt2...

Parameter counts for reference:
GPT-2: 124M parameters
GPT-2 Medium: 355M parameters
GPT-2 Large: 774M parameters
LLaMA-7B: 7B parameters
LLaMA-13B: 13B parameters


## Memory Requirements for Full Finetuning

For a model with N parameters in FP32:
- Model weights: 4N bytes
- Gradients: 4N bytes
- Optimizer states (Adam): 8N bytes
- **Total: ~16N bytes minimum**

For 7B parameters: ~112 GB just for training state!

In [10]:
def estimate_memory_gb(num_params_billions, precision="fp32"):
    """Estimate memory requirements for full finetuning"""
    params = num_params_billions * 1e9
    
    bytes_per_param = {"fp32": 4, "bf16": 2, "fp16": 2, "int8": 1}[precision]
    
    # Model weights
    model_memory = params * bytes_per_param
    
    # Gradients (same as model)
    gradient_memory = model_memory
    
    # Optimizer states (Adam: 2x model size for momentum and variance)
    optimizer_memory = 2 * params * 4  # Always in FP32
    
    # Activations (rough estimate: 20% of model size)
    activation_memory = model_memory * 0.2
    
    total_bytes = model_memory + gradient_memory + optimizer_memory + activation_memory
    total_gb = total_bytes / (1024**3)
    
    return {
        "model_gb": model_memory / (1024**3),
        "gradients_gb": gradient_memory / (1024**3),
        "optimizer_gb": optimizer_memory / (1024**3),
        "total_gb": total_gb
    }

# Example calculations
for model_size in [1, 7, 13, 70]:
    precision = 'bf16'
    mem = estimate_memory_gb(model_size, precision)
    print(f"\n{model_size}B model ({precision.upper()} full finetuning):")
    print(f"  Total memory: {mem['total_gb']:.1f} GB")


1B model (BF16 full finetuning):
  Total memory: 11.5 GB

7B model (BF16 full finetuning):
  Total memory: 80.8 GB

13B model (BF16 full finetuning):
  Total memory: 150.1 GB

70B model (BF16 full finetuning):
  Total memory: 808.4 GB


Helpful tool to estimate memory: https://rahulschand.github.io/gpu_poor/

# Part 3: LoRA - Low-Rank Adaptation

![Image](https://media.licdn.com/dms/image/v2/D5612AQEm0lBIuY3zHQ/article-cover_image-shrink_720_1280/article-cover_image-shrink_720_1280/0/1719589366257?e=1763596800&v=beta&t=pAm_4txNjjOg9A3yyBv2u_yC_yozQcAvlO5lGo0KXYc)

## Core Idea
Instead of updating all weights W, inject trainable low-rank decomposition:

**W' = W + ΔW = W + BA**

Where:
- W ∈ ℝ^(d×k) is the frozen pre-trained weight
- B ∈ ℝ^(d×r), A ∈ ℝ^(r×k) are trainable
- r << min(d, k) is the rank (typically 4-64)

## Key Benefits
1. **Fewer parameters**: Train only ~0.1-1% of original parameters
2. **Less memory**: No gradients for frozen weights
3. **Modular**: Can swap LoRA adapters without changing base model
4. **No inference overhead**: Merge ΔW into W after training

In [23]:
# Manual LoRA Implementation (Educational)
class LoRALayer(torch.nn.Module):
    """
    A simple LoRA layer implementation
    """
    def __init__(self, original_layer, rank=8, alpha=16):
        super().__init__()
        self.original_layer = original_layer
        self.rank = rank
        self.alpha = alpha
        
        # Freeze original weights
        for param in self.original_layer.parameters():
            param.requires_grad = False
        
        # Get dimensions
        if isinstance(original_layer, torch.nn.Linear):
            d, k = original_layer.out_features, original_layer.in_features
        else:
            raise ValueError("Only Linear layers supported in this demo")
        
        # Initialize LoRA matrices
        # A: Gaussian initialization
        self.lora_A = torch.nn.Parameter(torch.randn(rank, k) * 0.01)
        # B: Zero initialization (so ΔW starts at 0)
        self.lora_B = torch.nn.Parameter(torch.zeros(d, rank))
        
        # Scaling factor
        self.scaling = alpha / rank
    
    def forward(self, x):
        # Original forward pass
        original_out = self.original_layer(x)
        
        # LoRA forward pass: x @ A^T @ B^T
        lora_out = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        
        return original_out + lora_out
    
    def merge_weights(self):
        """Merge LoRA weights into original layer for inference"""
        if isinstance(self.original_layer, torch.nn.Linear):
            # ΔW = B @ A
            delta_w = (self.lora_B @ self.lora_A) * self.scaling
            self.original_layer.weight.data += delta_w

# Demonstrate parameter savings
d, k = 4096, 4096  # Typical transformer dimensions
rank = 8

original_params = d * k
lora_params = (d + k) * rank
reduction = (1 - lora_params / original_params) * 100

print(f"Original layer: {original_params:,} parameters")
print(f"LoRA layer: {lora_params:,} parameters")
print(f"Reduction: {reduction:.2f}%")
print(f"Parameter ratio: {lora_params / original_params:.4f}")

Original layer: 16,777,216 parameters
LoRA layer: 65,536 parameters
Reduction: 99.61%
Parameter ratio: 0.0039


## Why Does LoRA Work?

### Hypothesis: Update Matrix ΔW is Low-Rank

During finetuning, the weight update ΔW = W_finetuned - W_pretrained often has low intrinsic rank:
- Pre-trained models already capture rich representations
- Task adaptation requires adjusting in a lower-dimensional subspace
- Empirically verified: full finetuning ΔW has effective rank much lower than full rank

### Mathematical Intuition
If ΔW has effective rank r, then:
- ΔW ≈ UΣV^T where Σ has only r significant singular values
- We can approximate with rank-r decomposition: ΔW ≈ BA
- This is exactly what LoRA does!

In [5]:
# Simulate this with a toy example
np.random.seed(42)

# Create a "pre-trained" weight matrix
d, k = 100, 100
W_pretrained = np.random.randn(d, k) * 0.1

# Simulate a full finetuning update with low intrinsic rank
# (realistic: most adaptation happens in low-dimensional subspace)
true_rank = 5
U = np.random.randn(d, true_rank)
V = np.random.randn(true_rank, k)
Delta_W_true = U @ V * 0.01  # True low-rank update

W_finetuned = W_pretrained + Delta_W_true

# Now approximate with LoRA of different ranks
def lora_approximation_error(W_original, W_target, rank):
    """Compute approximation error for LoRA with given rank"""
    Delta_W = W_target - W_original
    
    # SVD of the update
    U, S, Vt = np.linalg.svd(Delta_W, full_matrices=False)
    
    # Truncate to rank r
    U_r = U[:, :rank]
    S_r = S[:rank]
    Vt_r = Vt[:rank, :]
    
    # Reconstruct
    Delta_W_approx = U_r @ np.diag(S_r) @ Vt_r
    
    # Error
    error = np.linalg.norm(Delta_W - Delta_W_approx, 'fro') / np.linalg.norm(Delta_W, 'fro')
    return error

print("LoRA approximation error for different ranks:")
for rank in [1, 2, 5, 10, 20]:
    error = lora_approximation_error(W_pretrained, W_finetuned, rank)
    print(f"  Rank {rank:2d}: {error:.4f} ({(1-error)*100:.1f}% of update captured)")

LoRA approximation error for different ranks:
  Rank  1: 0.7884 (21.2% of update captured)
  Rank  2: 0.6179 (38.2% of update captured)
  Rank  5: 0.0000 (100.0% of update captured)
  Rank 10: 0.0000 (100.0% of update captured)
  Rank 20: 0.0000 (100.0% of update captured)


## Which Layers to Apply LoRA?

Previously common strategies:
1. **Query & Value projections** (q_proj, v_proj): Most common, good balance
2. **All attention projections** (q, k, v, o): More parameters, better performance
3. **MLP layers**: Sometimes helps, especially for knowledge-intensive tasks

Trade-off: More LoRA layers → Better performance but more parameters


Under some conditions LORA can be equivalent to full fine-tuning. Latest research : https://thinkingmachines.ai/blog/lora/

# Part 4: Practical LoRA with PEFT

In [24]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

# Configuration
model_name = "gpt2"  # Small model for demo
dataset_name = "imdb"  # Sentiment analysis task

# Load model and tokenizer
print("Loading model and tokenizer...")
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print(f"Original model parameters: {count_parameters(model):,}")

Loading model and tokenizer...
Original model parameters: 124,439,808


In [25]:
# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                          # Rank
    lora_alpha=16,                # Scaling factor (typically 2*r)
    lora_dropout=0.05,            # Dropout for LoRA layers
    target_modules=["c_attn"],    # GPT-2 attention projection
    bias="none",                  # Don't train bias terms
)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


/home/robinhad/Projects/ucu-nlp-course/env/lib/python3.12/site-packages/peft/tuners/lora/layer.py:2174: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [26]:
# Prepare dataset (simplified for demo)
print("\nPreparing dataset...")
dataset = load_dataset(dataset_name, split="train[:1000]")  # Small subset for demo

def tokenize_function(examples):
    # For simplicity, just tokenize the text
    # In practice, you'd format this properly for your task
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)

print(f"Dataset size: {len(tokenized_dataset)} examples")


Preparing dataset...


Map: 100%|██████████| 1000/1000 [00:00<00:00, 11345.52 examples/s]

Dataset size: 1000 examples


In [27]:
# Training configuration
training_args = TrainingArguments(
    output_dir="./lora_output",
    num_train_epochs=1,           # Short demo
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,           # Higher LR for LoRA
    logging_steps=10,
    save_strategy="no",           # Don't save for demo
    report_to="none",
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# Note: Uncomment to actually train (takes time)
# print("\nStarting training...")
# trainer.train()

print("\n[Training code ready - uncomment trainer.train() to run]")


[Training code ready - uncomment trainer.train() to run]


## Merging and Saving LoRA Adapters

In [28]:
# After training, you can:

# 1. Save only the LoRA adapters (tiny files, ~few MB)
# model.save_pretrained("./my_lora_adapter")

# 2. Load adapters later
# from peft import PeftModel
# base_model = AutoModelForCausalLM.from_pretrained(model_name)
# model = PeftModel.from_pretrained(base_model, "./my_lora_adapter")

# 3. Merge adapters into base model for faster inference
# model = model.merge_and_unload()

print("LoRA adapter size vs full model:")
print("  Full model (GPT-2): ~500 MB")
print("  LoRA adapters (r=8): ~1-5 MB")
print("  Ratio: ~0.2-1% of full model size")

LoRA adapter size vs full model:
  Full model (GPT-2): ~500 MB
  LoRA adapters (r=8): ~1-5 MB
  Ratio: ~0.2-1% of full model size


In [29]:
# Experiment idea: Compare LoRA with different ranks

def compare_ranks_memory():
    """Compare memory usage for different LoRA ranks"""
    model_size = 7  # 7B model
    
    print("Memory comparison for 7B model:")
    print(f"Full finetuning: ~112 GB")
    
    for rank in [4, 8, 16, 32, 64, 128]:
        # Approximate LoRA memory
        # Main savings: no gradients for frozen weights
        # Still need optimizer states for LoRA params
        
        num_lora_layers = 32  # Typical for transformer
        d = 4096  # Hidden dimension
        
        lora_params = num_lora_layers * (d + d) * rank
        lora_params_billions = lora_params / 1e9
        
        # Memory for LoRA: model (frozen) + gradients (LoRA only) + optimizer (LoRA only)
        base_model_mem = 7 * 4  # 7B params * 4 bytes
        lora_grad_mem = lora_params_billions * 4
        lora_opt_mem = lora_params_billions * 8
        
        total_mem = base_model_mem + lora_grad_mem + lora_opt_mem
        
        print(f"  Rank {rank:3d}: ~{total_mem:.1f} GB ({lora_params_billions:.3f}B trainable params)")
    
    print("\n💡 Key insight: Even r=128 is far cheaper than full finetuning!")

compare_ranks_memory()

Memory comparison for 7B model:
Full finetuning: ~112 GB
  Rank   4: ~28.0 GB (0.001B trainable params)
  Rank   8: ~28.0 GB (0.002B trainable params)
  Rank  16: ~28.1 GB (0.004B trainable params)
  Rank  32: ~28.1 GB (0.008B trainable params)
  Rank  64: ~28.2 GB (0.017B trainable params)
  Rank 128: ~28.4 GB (0.034B trainable params)

💡 Key insight: Even r=128 is far cheaper than full finetuning!


# Part 5: QLoRA - Quantized LoRA

## Motivation
LoRA reduces trainable parameters, but base model still consumes memory:
- 7B model in FP16: ~14 GB just for weights
- 13B model in FP16: ~26 GB
- 70B model in FP16: ~140 GB (multiple GPUs needed)

**Solution: Quantize the base model to 4-bit!**

## QLoRA = LoRA + 4-bit Quantization

Key innovations:
1. **4-bit NormalFloat (NF4)**: Optimal quantization for normally distributed weights
2. **Double quantization**: Quantize the quantization constants
3. **Paged optimizers**: Handle memory spikes with CPU offloading

In [ ]:
# QLoRA memory savings
def qlora_memory_comparison():
    """Compare memory usage: Full FT vs LoRA vs QLoRA"""
    
    models = [7, 13, 70]
    
    for size in models:
        print(f"\n{size}B parameter model:")
        
        # Full finetuning (FP32)
        full_ft = estimate_memory_gb(size, "fp32")["total_gb"]
        
        # LoRA (FP16 base + FP16 adapters)
        lora_base = size * 2  # FP16 weights
        lora_adapters = 0.01 * size * 2  # ~1% params in FP16
        lora_opt = 0.01 * size * 8  # Optimizer for adapters
        lora_total = lora_base + lora_adapters + lora_opt
        
        # QLoRA (4-bit base + FP16 adapters)
        qlora_base = size * 0.5  # 4-bit weights
        qlora_adapters = 0.01 * size * 2  # ~1% params in FP16
        qlora_opt = 0.01 * size * 8  # Optimizer for adapters
        qlora_total = qlora_base + qlora_adapters + qlora_opt
        
        print(f"  Full FT:  ~{full_ft:.1f} GB")
        print(f"  LoRA:     ~{lora_total:.1f} GB ({lora_total/full_ft*100:.1f}% of Full FT)")
        print(f"  QLoRA:    ~{qlora_total:.1f} GB ({qlora_total/full_ft*100:.1f}% of Full FT)")
        print(f"  Savings:  {full_ft - qlora_total:.1f} GB saved!")

qlora_memory_comparison()

## 4-bit NormalFloat (NF4) Quantization

### Standard Quantization Problem
- Uniform quantization wastes bins on unlikely values
- Neural network weights follow ~Normal(0, σ)

### NF4 Solution
- Quantization levels optimized for normal distribution
- More bins near zero (where most weights are)
- Fewer bins in tails (where few weights exist)

Result: Better preservation of information with 4 bits!

In [30]:
import numpy as np
import matplotlib.pyplot as plt

# Visualize NF4 vs uniform quantization
def compare_quantization():
    """Compare uniform vs NF4 quantization"""
    
    # Generate normally distributed weights
    np.random.seed(42)
    weights = np.random.randn(10000)
    
    # Uniform quantization levels
    n_levels = 16  # 4-bit
    uniform_levels = np.linspace(-3, 3, n_levels)
    
    # NF4 quantization levels (simplified approximation)
    # More levels near zero, following inverse CDF of normal
    nf4_levels = np.array([
        -1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
        -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
        0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
        0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0
    ]) * 3  # Scale to similar range
    
    print("Quantization level distribution:")
    print(f"Uniform: Equal spacing")
    print(f"NF4: Denser near zero\n")
    
    # Count weights in each bin
    def count_bins(weights, levels):
        counts = np.zeros(len(levels) - 1)
        for i in range(len(levels) - 1):
            counts[i] = np.sum((weights >= levels[i]) & (weights < levels[i+1]))
        return counts
    
    uniform_usage = count_bins(weights, uniform_levels)
    nf4_usage = count_bins(weights, nf4_levels)
    
    print("Bin usage efficiency:")
    print(f"Uniform: {np.std(uniform_usage):.1f} std dev (less efficient)")
    print(f"NF4: {np.std(nf4_usage):.1f} std dev (more efficient)")

compare_quantization()

Quantization level distribution:
Uniform: Equal spacing
NF4: Denser near zero

Bin usage efficiency:
Uniform: 551.3 std dev (less efficient)
NF4: 299.5 std dev (more efficient)


## Practical QLoRA with PEFT

In [ ]:
"""
PEFT Fine-tuning of Mistral-7B for Sentiment Analysis
Uses LoRA (Low-Rank Adaptation) to fit under 16GB VRAM
"""

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import os

# Configuration
MODEL_NAME = "mistralai/Mistral-7B-v0.1"
OUTPUT_DIR = "./mistral-sentiment-lora"
MAX_LENGTH = 512

# LoRA Configuration - these parameters keep memory usage low
lora_config = LoraConfig(
    r=16,  # LoRA rank - higher = more parameters but better performance
    lora_alpha=32,  # LoRA scaling factor
    target_modules=[  # Which modules to apply LoRA to
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# 4-bit quantization config to reduce memory usage
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("\nLoading and preparing dataset...")
# Load IMDb sentiment dataset
dataset = load_dataset("imdb", split="train[:5000]")  # Using subset for quick demo
dataset = dataset.train_test_split(test_size=0.1, seed=42)


# Format data for instruction fine-tuning
def format_instruction(example):
    """Format the data as instruction-following task"""
    text = example["text"]
    label = "positive" if example["label"] == 1 else "negative"

    instruction = f"""### Instruction:
Analyze the sentiment of the following movie review and classify it as either positive or negative.

### Review:
{text[:500]}  

### Sentiment:
{label}"""

    return {"text": instruction}


# Apply formatting
formatted_dataset = dataset.map(
    format_instruction, remove_columns=dataset["train"].column_names
)


# Tokenization
def tokenize_function(examples):
    """Tokenize the text data"""
    tokenized = tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    tokenized["labels"] = tokenized["input_ids"].clone()
    return tokenized


tokenized_dataset = formatted_dataset.map(
    tokenize_function, batched=True, remove_columns=["text"], desc="Tokenizing dataset"
)

# Training arguments optimized for low memory
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,  # Effective batch size = 16
    learning_rate=2e-4,
    fp16=False,
    bf16=True,  # Use bfloat16 for better stability with LoRA
    logging_steps=10,
    logging_dir=f"{OUTPUT_DIR}/logs",
    save_strategy="epoch",
    eval_strategy="epoch",
    warmup_steps=50,
    optim="paged_adamw_8bit",  # 8-bit optimizer to save memory
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    report_to="none",  # Disable wandb/tensorboard for simplicity
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False  # We're doing causal language modeling
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
)

# Train the model
print("\nStarting training...")
print(f"Training on {len(tokenized_dataset['train'])} examples")
print(f"Evaluating on {len(tokenized_dataset['test'])} examples")

trainer.train()

# Save the fine-tuned model
print("\nSaving model...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\nTraining complete! Model saved to {OUTPUT_DIR}")

# Test inference
print("\n" + "=" * 50)
print("Testing the fine-tuned model:")
print("=" * 50)

model.eval()
test_review = """This movie was absolutely fantastic! The acting was superb, 
the plot was engaging, and I was on the edge of my seat the entire time."""

prompt = f"""### Instruction:
Analyze the sentiment of the following movie review and classify it as either positive or negative.

### Review:
{test_review}

### Sentiment:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)


## QLoRA Performance

### Key Results (from QLoRA paper):
- 4-bit QLoRA matches 16-bit LoRA performance
- Minimal quality degradation vs full finetuning
- Enables finetuning 65B models on single 48GB GPU

### Trade-offs:
- 4x less memory for base model
- Same performance as LoRA
- Slightly slower training (quantization overhead)
- Requires bitsandbytes library (not all platforms)

# Comparison Summary & Best Practices (5-10 minutes)

## Best Practices & Tips

# Always go normal first!


But if you have to:

### Choosing a Method:
1. **Start with QLoRA (r=16)** - Best bang for buck
2. **If accuracy matters most** - Try LoRA with r=64 or full FT
3. **If memory is tight** - QLoRA with r=8
4. **For production** - Train with LoRA/QLoRA, merge for inference

### Hyperparameter Guidelines:
- **Rank (r)**: Start with 16, increase if underfitting (8, 16, 32, 64)
- **Alpha**: Typically 2*r (scaling factor)
- **Learning rate**: 1e-4 to 3e-4 (higher than full FT)
- **Target modules**: q_proj, v_proj minimum; add k_proj, o_proj, mlp for harder tasks
- **Dropout**: 0.05-0.1 for regularization

### Common Pitfalls:
- Too low rank → Underfitting
- Too high LR → Instability
- Wrong target modules → Poor performance
- Not enough training steps → Doesn't converge

# Additional Resources

## Papers to Read:
1. **LoRA**: Hu et al. (2021) - "LoRA: Low-Rank Adaptation of Large Language Models"
2. **QLoRA**: Dettmers et al. (2023) - "QLoRA: Efficient Finetuning of Quantized LLMs"
3. **DoRA**: Liu et al. (2024) - "DoRA: Weight-Decomposed Low-Rank Adaptation"
4. **Research on LoRA = Full FT**: Check recent papers on rank requirements

## Libraries:
- **PEFT** (Parameter-Efficient Fine-Tuning): https://github.com/huggingface/peft
- **bitsandbytes**: https://github.com/TimDettmers/bitsandbytes
- **trl** (Transformer Reinforcement Learning): For RLHF with LoRA

## Tips for Further Learning:
- Experiment with different model sizes
- Try different tasks (classification, generation, QA)
- Monitor training metrics closely
- Join communities: HuggingFace forums, Reddit r/LocalLLaMA